# EWS Multivariate Anomaly Detection — Full Pipeline

**Akis:**
1. Oracle'a tablo kur ve veri yukle
2. Training verisini Oracle'dan cek
3. Train/Test split ile model egit ve performans olc
4. Scoring verisini Oracle'dan cek, skorla
5. Sonuclari Oracle'a yaz
6. Is birimi raporu uret

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

from config import ALL_FEATURES, FEATURE_LABELS
from config import PAYMENT_FEATURES, UTILIZATION_FEATURES, TRANSACTION_FEATURES, BALANCE_FEATURES, TREND_FEATURES
from oracle_config import get_connection, TABLE_TRAIN, TABLE_SCORING, TABLE_RESULTS, TABLE_DETAILS, SCHEMA
from model import ExplainableEnsemble

print(f'Feature sayisi: {len(ALL_FEATURES)}')

---
## 1. Oracle Setup — Tablo Olustur ve Veri Yukle

In [ ]:
# Tablolari olustur ve veri yukle
# Bu hucre sadece ilk kurulumda calistirilir

from setup_oracle import main as setup_main
setup_main()

In [ ]:
# Dogrulama: tablolardaki kayit sayilari
conn = get_connection()
cursor = conn.cursor()

for tbl in [TABLE_TRAIN, TABLE_SCORING]:
    cursor.execute(f'SELECT COUNT(*) FROM {tbl}')
    cnt = cursor.fetchone()[0]
    print(f'{tbl}: {cnt} rows')

# Training split dagilimi
cursor.execute(f'SELECT SPLIT_FLAG, COUNT(*) FROM {TABLE_TRAIN} GROUP BY SPLIT_FLAG')
for flag, cnt in cursor.fetchall():
    print(f'  {flag}: {cnt}')

cursor.close()
conn.close()

---
## 2. Training Verisini Oracle'dan Cek

In [ ]:
conn = get_connection()

# TRAIN seti
feature_cols_upper = ', '.join([f.upper() for f in ALL_FEATURES])
sql_train = f"""
    SELECT CUSTOMER_ID, {feature_cols_upper}
    FROM {TABLE_TRAIN}
    WHERE SPLIT_FLAG = 'TRAIN'
"""
train_df = pd.read_sql(sql_train, conn)
train_df.columns = ['customer_id'] + ALL_FEATURES

# TEST seti (holdout — performans olcumu icin)
sql_test = f"""
    SELECT CUSTOMER_ID, {feature_cols_upper}
    FROM {TABLE_TRAIN}
    WHERE SPLIT_FLAG = 'TEST'
"""
test_df = pd.read_sql(sql_test, conn)
test_df.columns = ['customer_id'] + ALL_FEATURES

conn.close()

print(f'Train: {train_df.shape}')
print(f'Test (holdout): {test_df.shape}')

In [ ]:
# EDA — onemli feature dagilimları
key_feats = ['utilization_ratio', 'days_past_due', 'min_payment_only_count_6m',
             'payment_to_minimum_ratio', 'outstanding_balance', 'avg_checking_balance_3m']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, feat in zip(axes.flat, key_feats):
    ax.hist(train_df[feat], bins=50, alpha=0.7, color='#3498db', edgecolor='white')
    ax.set_title(FEATURE_LABELS.get(feat, feat), fontsize=10)
    ax.axvline(train_df[feat].mean(), color='red', linestyle='--', alpha=0.7)
fig.suptitle('Training Data — Temel Feature Dagilimları', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Korelasyon matrisi
corr = train_df[ALL_FEATURES].corr()

fig, ax = plt.subplots(figsize=(16, 13))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.3, ax=ax,
            xticklabels=[f[:14] for f in ALL_FEATURES],
            yticklabels=[f[:14] for f in ALL_FEATURES])
ax.set_title('Feature Korelasyon Matrisi', fontsize=14)
plt.tight_layout()
plt.show()

---
## 3. Model Egitimi (Train) + Performans Olcumu (Test)

### Unsupervised modelde performans nasil olculur?

Label yok — precision/recall direkt hesaplanamaz. Bunun yerine:

| Metrik | Ne Olcer | Nasil Yorumlanir |
|--------|----------|------------------|
| **Reconstruction Stability** | Train vs Test hata dagilimi benzer mi? | KS test p>0.05 → model stabil |
| **Score Distribution** | Train vs Test skor dagilimi benzer mi? | Cok farkli → overfit |
| **Anomaly Separation** | Yuksek skorlular dusuklerden ayrilabiliyor mu? | Bimodal dagilim iyi |
| **Feature Consistency** | Ayni feature'lar mi one cikiyor? | Top-5 benzer → stabil model |
| **Threshold Sensitivity** | Esik degisince alert sayisi ne kadar degisiyor? | Cok hassas → esik belirsiz |

In [ ]:
# Modeli SADECE train seti ile egit
model = ExplainableEnsemble()
model.fit(train_df)

In [ ]:
# ═══════════════════════════════════════════════════════════
# PERFORMANS METRIK 1: Reconstruction Stability
# Train ve Test'teki AE reconstruction error benzer mi?
# ═══════════════════════════════════════════════════════════

from scipy.stats import ks_2samp

X_train = model.scaler.transform(train_df[ALL_FEATURES].fillna(0))
X_test = model.scaler.transform(test_df[ALL_FEATURES].fillna(0))

train_ae_err = model._ae_total_error(X_train)
test_ae_err = model._ae_total_error(X_test)

ks_stat, ks_pval = ks_2samp(train_ae_err, test_ae_err)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(train_ae_err, bins=60, alpha=0.6, color='#3498db', label=f'Train (n={len(train_ae_err)})', density=True, edgecolor='white')
axes[0].hist(test_ae_err, bins=60, alpha=0.6, color='#e74c3c', label=f'Test (n={len(test_ae_err)})', density=True, edgecolor='white')
axes[0].set_title(f'AE Reconstruction Error\nKS stat={ks_stat:.4f}, p={ks_pval:.4f}', fontsize=12)
axes[0].legend()
axes[0].set_xlabel('Reconstruction Error')

# Mahalanobis
train_md = model._mahal_distances(X_train)
test_md = model._mahal_distances(X_test)
ks2_stat, ks2_pval = ks_2samp(train_md, test_md)

axes[1].hist(train_md, bins=60, alpha=0.6, color='#3498db', label='Train', density=True, edgecolor='white')
axes[1].hist(test_md, bins=60, alpha=0.6, color='#e74c3c', label='Test', density=True, edgecolor='white')
axes[1].set_title(f'Mahalanobis Distance\nKS stat={ks2_stat:.4f}, p={ks2_pval:.4f}', fontsize=12)
axes[1].legend()
axes[1].set_xlabel('Distance')

plt.tight_layout()
plt.show()

print('Yorum:')
if ks_pval > 0.05:
    print(f'  ✓ AE: Train ve Test dagilimi istatistiksel olarak BENZER (p={ks_pval:.4f} > 0.05)')
    print(f'    Model overfit YAPMAMIS — yeni veri uzerinde tutarli calisiyor.')
else:
    print(f'  ⚠ AE: Train ve Test dagilimi FARKLI (p={ks_pval:.4f} < 0.05)')
    print(f'    Model overfit olmus olabilir. Epoch sayisi veya latent dim ayarlama gerekebilir.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# PERFORMANS METRIK 2: Score Distribution Stability
# Train ve Test uzerindeki ensemble skor dagilimi benzer mi?
# ═══════════════════════════════════════════════════════════

train_results = model.predict(train_df, top_n=3)
test_results = model.predict(test_df, top_n=3)

ks3_stat, ks3_pval = ks_2samp(train_results['anomaly_score'], test_results['anomaly_score'])

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(train_results['anomaly_score'], bins=50, alpha=0.5, color='#3498db',
        label=f'Train (n={len(train_results)})', density=True, edgecolor='white')
ax.hist(test_results['anomaly_score'], bins=50, alpha=0.5, color='#e74c3c',
        label=f'Test (n={len(test_results)})', density=True, edgecolor='white')

for val, lbl, clr in [(60,'SARI','#f1c40f'), (75,'TURUNCU','#e67e22'), (90,'KIRMIZI','#e74c3c')]:
    ax.axvline(val, color=clr, linestyle='--', alpha=0.6)

ax.set_title(f'Ensemble Score: Train vs Test\nKS stat={ks3_stat:.4f}, p={ks3_pval:.4f}', fontsize=13)
ax.set_xlabel('Anomaly Score (0-100)')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

# Bant dagilim karsilastirmasi
print('\nBant dagilimi karsilastirmasi:')
for band in ['KIRMIZI','TURUNCU','SARI','NORMAL']:
    tr_pct = (train_results['alert_band']==band).mean()*100
    te_pct = (test_results['alert_band']==band).mean()*100
    diff = abs(tr_pct - te_pct)
    flag = '  ✓' if diff < 3 else '  ⚠' 
    print(f'  {band:10s}  Train: %{tr_pct:.1f}  Test: %{te_pct:.1f}  Fark: {diff:.1f}pp {flag}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# PERFORMANS METRIK 3: Feature Contribution Consistency
# Train ve Test'te ayni feature'lar mi one cikiyor?
# ═══════════════════════════════════════════════════════════

def top_contributing_features(results_df, n=10):
    """Alert verilen musterilerde en sik top-1 neden olan feature'lar."""
    alerts = results_df[results_df['alert_band'].isin(['KIRMIZI','TURUNCU','SARI'])]
    feat_counts = {}
    for _, row in alerts.iterrows():
        if row['detay']:
            first_feat = list(row['detay'].keys())[0]
            feat_counts[first_feat] = feat_counts.get(first_feat, 0) + 1
    return pd.Series(feat_counts).sort_values(ascending=False).head(n)

train_top = top_contributing_features(train_results)
test_top = top_contributing_features(test_results)

print('En sik "ana etken" olan feature\'lar:')
print(f'\n{"Train":>35s} | {"Test":>35s}')
print('-'*75)
for i in range(min(8, max(len(train_top), len(test_top)))):
    tr = f"{train_top.index[i]:25s} ({train_top.iloc[i]:3d})" if i < len(train_top) else ''
    te = f"{test_top.index[i]:25s} ({test_top.iloc[i]:3d})" if i < len(test_top) else ''
    print(f'{tr:>35s} | {te:>35s}')

# Overlap
overlap = set(train_top.head(5).index) & set(test_top.head(5).index)
print(f'\nTop-5 overlap: {len(overlap)}/5 — ', end='')
if len(overlap) >= 4:
    print('✓ Cok tutarli. Model stabil.')
elif len(overlap) >= 3:
    print('~ Makul. Kucuk farklar beklenir.')
else:
    print('⚠ Dusuk tutarlilik. Model kararsiz olabilir.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# PERFORMANS METRIK 4: Threshold Sensitivity
# Esik degistikce alert sayisi nasil degisiyor?
# ═══════════════════════════════════════════════════════════

thresholds = list(range(40, 100, 2))
train_counts = [(test_results['anomaly_score'] >= t).sum() for t in thresholds]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(thresholds, train_counts, 'o-', color='#3498db', linewidth=2, markersize=4)

for val, lbl, clr in [(60,'SARI','#f1c40f'), (75,'TURUNCU','#e67e22'), (90,'KIRMIZI','#e74c3c')]:
    ax.axvline(val, color=clr, linestyle='--', alpha=0.6)
    cnt = (test_results['anomaly_score'] >= val).sum()
    ax.text(val+1, cnt, f'{lbl}\n{cnt}', fontsize=9, color=clr)

ax.set_xlabel('Esik Degeri', fontsize=12)
ax.set_ylabel('Alert Sayisi', fontsize=12)
ax.set_title('Threshold Sensitivity — Esik vs Alert Sayisi (Test Seti)', fontsize=13)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Yorum:')
print('  Esik-alert egrisinde keskin bir dirsek (elbow) varsa → dogal ayrisma noktasi.')
print('  Duz egri → net bir ayrisma yok, esik secimi is birimiyle birlikte yapilmali.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# PERFORMANS METRIK 5: Istatistiksel Ozet
# ═══════════════════════════════════════════════════════════

print('='*65)
print('MODEL PERFORMANS OZETI')
print('='*65)
print(f'\n  Train seti: {len(train_df)} musteri')
print(f'  Test seti:  {len(test_df)} musteri')
print(f'  Feature:    {len(ALL_FEATURES)} degisken')
print(f'\n  AE final loss:     {model._ae_total_error(X_train).mean():.4f}')
print(f'  AE test loss:      {model._ae_total_error(X_test).mean():.4f}')
print(f'  AE loss ratio:     {model._ae_total_error(X_test).mean() / model._ae_total_error(X_train).mean():.3f}x')
print(f'    (1.0a yakin = iyi, >1.5 = overfit)')

print(f'\n  Reconstruction Stability (KS): p={ks_pval:.4f} {"✓" if ks_pval > 0.05 else "⚠"}')
print(f'  Score Stability (KS):          p={ks3_pval:.4f} {"✓" if ks3_pval > 0.05 else "⚠"}')
print(f'  Feature Consistency:           {len(overlap)}/5 overlap {"✓" if len(overlap)>=4 else "~"}')

print(f'\n  Test seti bant dagilimi:')
for band in ['KIRMIZI','TURUNCU','SARI','NORMAL']:
    cnt = (test_results['alert_band']==band).sum()
    pct = cnt / len(test_results) * 100
    print(f'    {band:10s}: {cnt:5d} (%{pct:.1f})')

---
## 4. Scoring — Bugunun Verisini Oracle'dan Cek ve Skorla

In [ ]:
# Scoring verisini Oracle'dan cek
conn = get_connection()

sql_scoring = f"""
    SELECT CUSTOMER_ID, {feature_cols_upper}
    FROM {TABLE_SCORING}
"""
scoring_df = pd.read_sql(sql_scoring, conn)
scoring_df.columns = ['customer_id'] + ALL_FEATURES
conn.close()

print(f'Scoring verisi: {scoring_df.shape}')
scoring_df.head()

In [ ]:
# Skorla
results = model.predict(scoring_df, top_n=3)

print(f'Skorlanan musteri: {len(results)}')
print(f'\nBant dagilimi:')
print(results['alert_band'].value_counts().sort_index())
print(f'\nSkor istatistikleri:')
print(results['anomaly_score'].describe().round(1))

In [ ]:
# Skor dagilimi
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'NORMAL':'#2ecc71', 'SARI':'#f1c40f', 'TURUNCU':'#e67e22', 'KIRMIZI':'#e74c3c'}
for band in ['NORMAL','SARI','TURUNCU','KIRMIZI']:
    s = results[results['alert_band']==band]['anomaly_score']
    axes[0].hist(s, bins=40, alpha=0.6, color=colors[band], label=f'{band} ({len(s)})', edgecolor='white')
axes[0].set_title('Anomaly Score Dagilimi', fontsize=12)
axes[0].set_xlabel('Score')
axes[0].legend()

axes[1].scatter(results['ae_score'], results['if_score'], c=results['anomaly_score'],
                cmap='RdYlGn_r', alpha=0.4, s=15)
axes[1].set_xlabel('AE Score')
axes[1].set_ylabel('IF Score')
axes[1].set_title('AE vs IF', fontsize=12)
plt.colorbar(axes[1].collections[0], ax=axes[1], label='Ensemble')
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 alert — human readable
print('='*80)
print('TOP 10 ALERT')
print('='*80)

for _, row in results.head(10).iterrows():
    icon = {'KIRMIZI':'\U0001f534','TURUNCU':'\U0001f7e0','SARI':'\U0001f7e1'}.get(row['alert_band'],'\u26aa')
    print(f"\n{icon} {row['customer_id']}  |  Skor: {row['anomaly_score']}  |  {row['alert_band']}")
    print(f"   AE: {row['ae_score']}  |  IF: {row['if_score']}  |  MD: {row['md_score']}")
    for feat, d in row['detay'].items():
        ico = '\u2191' if d['degisim_pct'] > 0 else '\u2193'
        print(f"   \u2022 {d['label']}: {d['beklenen']} \u2192 {d['gerceklesen']} ({ico}%{abs(d['degisim_pct']):.0f}, katki %{d['katki_pct']:.0f})")

---
## 5. Sonuclari Oracle'a Yaz

In [ ]:
from datetime import date

conn = get_connection()
cursor = conn.cursor()
today = date.today()

# ── Alert Results tablosu ──
alert_rows = []
for _, row in results.iterrows():
    # Top 3 neden
    nedenler = []
    for feat, d in row['detay'].items():
        ico = '↑' if d['degisim_pct'] > 0 else '↓'
        nedenler.append(f"{d['label']}: {d['beklenen']}→{d['gerceklesen']} ({ico}%{abs(d['degisim_pct']):.0f}, katki%{d['katki_pct']:.0f})")
    while len(nedenler) < 3:
        nedenler.append(None)

    alert_rows.append([
        row['customer_id'], today,
        float(row['anomaly_score']), str(row['alert_band']),
        float(row['ae_score']), float(row['if_score']), float(row['md_score']),
        nedenler[0], nedenler[1], nedenler[2]
    ])

sql_res = f"""
    INSERT INTO {TABLE_RESULTS}
    (CUSTOMER_ID, SCORING_DATE, ANOMALY_SCORE, ALERT_BAND, AE_SCORE, IF_SCORE, MD_SCORE,
     NEDEN_1, NEDEN_2, NEDEN_3)
    VALUES (:1,:2,:3,:4,:5,:6,:7,:8,:9,:10)
"""

for i in range(0, len(alert_rows), 500):
    cursor.executemany(sql_res, alert_rows[i:i+500])

print(f'EWS_ALERT_RESULTS: {len(alert_rows)} rows inserted')

# ── Alert Details tablosu (sadece alert verilenler) ──
alerts_only = results[results['alert_band'].isin(['KIRMIZI','TURUNCU','SARI'])]
detail_rows = []
for _, row in alerts_only.iterrows():
    for sira, (feat, d) in enumerate(row['detay'].items(), 1):
        detail_rows.append([
            row['customer_id'], today, feat, d['label'],
            float(d['beklenen']), float(d['gerceklesen']),
            float(d['degisim_pct']), float(d['katki_pct']), sira
        ])

sql_det = f"""
    INSERT INTO {TABLE_DETAILS}
    (CUSTOMER_ID, SCORING_DATE, FEATURE_NAME, FEATURE_LABEL,
     BEKLENEN, GERCEKLESEN, DEGISIM_PCT, KATKI_PCT, SIRA)
    VALUES (:1,:2,:3,:4,:5,:6,:7,:8,:9)
"""

for i in range(0, len(detail_rows), 500):
    cursor.executemany(sql_det, detail_rows[i:i+500])

conn.commit()
cursor.close()
conn.close()

print(f'EWS_ALERT_DETAILS: {len(detail_rows)} rows inserted')
print('\nOracle\'a yazildi ✓')

In [ ]:
# Dogrulama: Oracle'dan oku
conn = get_connection()

verify = pd.read_sql(f"""
    SELECT CUSTOMER_ID, ANOMALY_SCORE, ALERT_BAND, NEDEN_1, NEDEN_2, NEDEN_3
    FROM {TABLE_RESULTS}
    WHERE ALERT_BAND = 'KIRMIZI'
    ORDER BY ANOMALY_SCORE DESC
    FETCH FIRST 10 ROWS ONLY
""", conn)

conn.close()
print(f'Oracle\'dan KIRMIZI alertler ({len(verify)} rows):')
verify

---
## 6. Is Birimi Raporu

In [ ]:
# Excel raporu
import os
os.makedirs('output', exist_ok=True)

alert_df = results[results['alert_band'].isin(['KIRMIZI','TURUNCU','SARI'])].copy()

report_rows = []
for _, row in alert_df.iterrows():
    r = {
        'Musteri': row['customer_id'],
        'Skor': row['anomaly_score'],
        'Bant': row['alert_band'],
        'AE': row['ae_score'],
        'IF': row['if_score'],
        'MD': row['md_score'],
    }
    for i, (feat, d) in enumerate(row['detay'].items(), 1):
        ico = '↑' if d['degisim_pct'] > 0 else '↓'
        r[f'Neden_{i}'] = f"{d['label']}: {d['beklenen']}→{d['gerceklesen']} ({ico}%{abs(d['degisim_pct']):.0f})"
        r[f'Katki_{i}'] = f"%{d['katki_pct']}"
    report_rows.append(r)

report = pd.DataFrame(report_rows)
report.to_excel('output/haftalik_alert_raporu.xlsx', index=False, sheet_name='EWS Alerts')

print(f'Toplam alert: {len(report)}')
print(f'  KIRMIZI: {len(report[report["Bant"]=="KIRMIZI"])}')
print(f'  TURUNCU: {len(report[report["Bant"]=="TURUNCU"])}')
print(f'  SARI:    {len(report[report["Bant"]=="SARI"])}')
print(f'\nKaydedildi: output/haftalik_alert_raporu.xlsx')

---
## 7. Validation — Enjekte Edilen Anomalileri Yakalıyor muyuz?

Sentetik veri olduğu için anomali label'ları biliniyor. Gerçek projede bu kısım iş biriminin feedback'i ile yapılacak.

In [ ]:
# Anomali labels'i yukle (sadece validation — production'da olmaz)
from generate_data import generate_scoring_data
_, labels = generate_scoring_data()

eval_df = results.merge(labels, on='customer_id')

# Bant bazli precision
print('='*65)
print('VALIDATION (sentetik label ile)')
print('='*65)

for band in ['KIRMIZI','TURUNCU','SARI']:
    in_band = eval_df[eval_df['alert_band']==band]
    n = len(in_band)
    true_pos = in_band['is_anomaly'].sum()
    prec = true_pos / n * 100 if n > 0 else 0
    print(f'\n  {band}: {n} alert, {true_pos} gercek anomali (precision: %{prec:.1f})')
    by_type = in_band[in_band['is_anomaly']].groupby('anomaly_type').size()
    for t, c in by_type.items():
        print(f'    {t}: {c}')

# Recall
print(f'\n  RECALL:')
alerted = eval_df[eval_df['alert_band'].isin(['KIRMIZI','TURUNCU','SARI'])]
for atype in ['A_UNIVARIATE','B_MULTIVARIATE','C_SUBTLE_DRIFT']:
    total = (eval_df['anomaly_type']==atype).sum()
    caught = (alerted['anomaly_type']==atype).sum()
    print(f'    {atype}: {caught}/{total} (%{caught/total*100:.1f})')

total_a = eval_df['is_anomaly'].sum()
total_c = alerted['is_anomaly'].sum()
print(f'    TOPLAM: {total_c}/{total_a} (%{total_c/total_a*100:.1f})')

In [ ]:
# Anomali tiplerine gore skor dagilimi
fig, ax = plt.subplots(figsize=(12, 5))

type_colors = {'NORMAL':'#2ecc71', 'A_UNIVARIATE':'#e74c3c',
               'B_MULTIVARIATE':'#e67e22', 'C_SUBTLE_DRIFT':'#9b59b6'}

for atype, color in type_colors.items():
    s = eval_df[eval_df['anomaly_type']==atype]['anomaly_score']
    ax.hist(s, bins=40, alpha=0.5, color=color, label=f'{atype} (n={len(s)})', density=True)

for val, lbl, clr in [(60,'SARI','#f1c40f'),(75,'TURUNCU','#e67e22'),(90,'KIRMIZI','#e74c3c')]:
    ax.axvline(val, color=clr, linestyle='--', alpha=0.6)

ax.set_xlabel('Anomaly Score')
ax.set_title('Anomali Tipi vs Skor Dagilimi', fontsize=13)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Tek musteri deep dive
red = eval_df[(eval_df['alert_band']=='KIRMIZI') & (eval_df['is_anomaly'])]
sample = red.iloc[0]
cust_id = sample['customer_id']

print(f'\n{"="*60}')
print(f'DEEP DIVE: {cust_id}')
print(f'{"="*60}')
print(f'Tip: {sample["anomaly_type"]}')
print(f'Skor: {sample["anomaly_score"]} | Bant: {sample["alert_band"]}')
print(f'AE: {sample["ae_score"]} | IF: {sample["if_score"]} | MD: {sample["md_score"]}')

# 3 model katki paylari
cust_row = scoring_df[scoring_df['customer_id']==cust_id]
X_c = model.scaler.transform(cust_row[ALL_FEATURES].fillna(0))

ae_c = model._ae_contribution(X_c)[0]
if_c = model._if_contribution(X_c)[0]
md_c = model._md_contribution(X_c)[0]
unified = 0.5*ae_c + 0.3*if_c + 0.2*md_c

contrib = pd.DataFrame({
    'Feature': ALL_FEATURES,
    'Label': [FEATURE_LABELS.get(f,f) for f in ALL_FEATURES],
    'AE%': (ae_c*100).round(1),
    'IF%': (if_c*100).round(1),
    'MD%': (md_c*100).round(1),
    'Birlesik%': (unified*100).round(1),
}).sort_values('Birlesik%', ascending=False)

print(f'\nTop 8 Feature Katki:')
print(contrib.head(8).to_string(index=False))

# Gorsel
top8 = contrib.head(8)
fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(top8))
w = 0.22
ax.bar(x-w, top8['AE%'], w, label='AE', color='#3498db', alpha=0.8)
ax.bar(x, top8['IF%'], w, label='IF', color='#e67e22', alpha=0.8)
ax.bar(x+w, top8['MD%'], w, label='MD', color='#9b59b6', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([l[:18] for l in top8['Label']], rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Katki %')
ax.set_title(f'{cust_id} — 3 Model Feature Katkilari', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()